## Setup

In [ ]:
# terminal
# python -m venv .venv
# source .venv/bin/activate
# pip install pymc

In [2]:
# import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import os

## Generate synthetic data
Data sets that contain product catalog and elasticities

In [48]:
# --------------------------------------------------------------------------
# Create product catalog
# --------------------------------------------------------------------------

categories = {
    "Sedan": {
        "brands": ["Honda", "Hyundai", "Toyota"],
        "elasticity_mean": -0.45,
        "elasticity_std": 0.10,
        "cross_price_mean": 0.35,   # highly substitutable
        "cross_price_std": 0.02,
    },
    "SmallSUV": {
        "brands": ["Volkswagen", "Mazda", "Chevrolet"],
        "elasticity_mean": -1.0,
        "elasticity_std": 0.15,
        "cross_price_mean": 0.20,   # moderate substitution
        "cross_price_std": 0.06,
    },
    "LargeSUV": {
        "brands": ["Tesla", "Jeep", "Porsche"],
        "elasticity_mean": -1.2,
        "elasticity_std": 0.30,
        "cross_price_mean": 0,   # not substitutable
        "cross_price_std": 0.08,
    },
}

states = {
    "Florida": {"elasticity_shift": 0.0},
    "NewYork":      {"elasticity_shift": -0.15},
}

N_WEEKS = 104  # 2 years of weekly data

# Control variable coefficients (shared across all SKUs)
TRUE_BETA_SEASON = 0.15
TRUE_SIGMA = 0.12

In [ ]:
# save the product catalog to a DataFrame
folder_path = os.path.join('data')
if os.path.exists(folder_path):
    print(f"Folder '{folder_path}' already exists.")
else:
  os.makedirs(folder_path)
  print(f'folder {folder_path} created!')


Folder 'data' already exists.


In [49]:
# --------------------------------------------------------------------------
# Generate true parameters: own-price + cross-price elasticities
# --------------------------------------------------------------------------
rng = np.random.default_rng(seed=42)
true_params = {}
product_list = []

for cat_name, cat_info in categories.items():
    for product_name in cat_info["brands"]:
        for state_name,state_info in states.items():
            product_id = f"{product_name} ({state_name})"

            true_elasticity = (
                cat_info["elasticity_mean"]
                + rng.normal(0, cat_info["elasticity_std"])
                + state_info["elasticity_shift"]
            )

            true_intercept = rng.uniform(9.0, 13.0)

            true_params[product_id] = {
                "category": cat_name,
                "state": state_name,
                "product": product_name,
                "elasticity": true_elasticity,
                "intercept": true_intercept,
            }
            product_list.append(product_id)

# Build cross-price elasticity matrix: gamma[i][j] for j in same cat+region
# Each pair gets its own coefficient, drawn from the category-level distribution
true_cross_price = {}  # dict of {product_id: {other_product_id: gamma_ij}}

for product_id, params in true_params.items():
    cat = params["category"]
    state = params["state"]
    cat_info = categories[cat]

    competitors = [
        sid for sid, p in true_params.items()
        if p["category"] == cat and p["state"] == state and sid != product_id
    ]

    cross_coeffs = {}
    for comp_id in competitors:
        # Each pair (i, j) gets its own coefficient, drawn from category distribution
        gamma_ij = max(0, rng.normal(cat_info["cross_price_mean"], cat_info["cross_price_std"]))
        cross_coeffs[comp_id] = gamma_ij

    true_cross_price[product_id] = cross_coeffs

# Display
print(f"Total Product × State combinations: {len(product_list)}\n")

print("Own-price elasticities:")
for sid in product_list:
    p = true_params[sid]
    print(f"  {sid:35s}  [{p['category']:11s}]  ε = {p['elasticity']:.3f}")

print(f"\nCross-price elasticity matrix (per Product pair):")
for sid in product_list:
    cross = true_cross_price[sid]
    if cross:
        for comp_id, gamma in cross.items():
            comp_short = true_params[comp_id]["product"]
            own_short = true_params[sid]["product"]
            print(f"  {own_short:16s} ← {comp_short:16s} ({true_params[sid]['state']:7s})  γ = {gamma:.3f}")

Total Product × State combinations: 18

Own-price elasticities:
  Honda (Florida)                      [Sedan      ]  ε = -0.420
  Honda (NewYork)                      [Sedan      ]  ε = -0.525
  Hyundai (Florida)                    [Sedan      ]  ε = -0.645
  Hyundai (NewYork)                    [Sedan      ]  ε = -0.587
  Toyota (Florida)                     [Sedan      ]  ε = -0.452
  Toyota (NewYork)                     [Sedan      ]  ε = -0.512
  Volkswagen (Florida)                 [SmallSUV   ]  ε = -0.990
  Volkswagen (NewYork)                 [SmallSUV   ]  ε = -1.080
  Mazda (Florida)                      [SmallSUV   ]  ε = -0.945
  Mazda (NewYork)                      [SmallSUV   ]  ε = -1.018
  Chevrolet (Florida)                  [SmallSUV   ]  ε = -1.028
  Chevrolet (NewYork)                  [SmallSUV   ]  ε = -0.967
  Tesla (Florida)                      [LargeSUV   ]  ε = -1.328
  Tesla (NewYork)                      [LargeSUV   ]  ε = -1.190
  Jeep (Florida)          

In [61]:
# save product elasticities dataframe to csv
true_params_df = pd.DataFrame.from_dict(true_params, orient="index")
true_cross_price_df = pd.DataFrame.from_dict(true_cross_price, orient="index")
product_elasticities_df = true_params_df.join(true_cross_price_df, how='left')
product_elasticities_df.to_csv('data/product_elasticities_df.csv', index=False)

In [62]:
product_elasticities_df

,category,state,product,elasticity,intercept,Hyundai (Florida),Toyota (Florida),Hyundai (NewYork),Toyota (NewYork),Honda (Florida),...,Mazda (NewYork),Chevrolet (NewYork),Volkswagen (Florida),Volkswagen (NewYork),Jeep (Florida),Porsche (Florida),Jeep (NewYork),Porsche (NewYork),Tesla (Florida),Tesla (NewYork)
Honda (Florida),Sedan,Florida,Honda,-0.419528,10.755514,0.347721,0.333197,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Honda (NewYork),Sedan,NewYork,Honda,-0.524955,11.789472,NaN,NaN,0.333510,0.363012,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hyundai (Florida),Sedan,Florida,Hyundai,-0.645104,12.902489,NaN,0.360863,NaN,NaN,0.364865,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hyundai (NewYork),Sedan,NewYork,Hyundai,-0.587216,12.144257,NaN,NaN,NaN,0.354643,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Toyota (Florida),Sedan,Florida,Toyota,-0.451680,10.801544,0.354374,NaN,NaN,NaN,0.352334,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Toyota (NewYork),Sedan,NewYork,Toyota,-0.512060,12.707060,NaN,NaN,0.354472,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Volkswagen (Florida),SmallSUV,Florida,Volkswagen,-0.990095,12.291046,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Volkswagen (NewYork),SmallSUV,NewYork,Volkswagen,-1.079874,9.908955,NaN,NaN,NaN,NaN,NaN,...,0.217347,0.237877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Mazda (Florida),SmallSUV,Florida,Mazda,-0.944687,9.255269,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.112571,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Mazda (NewYork),SmallSUV,NewYork,Mazda,-1.018232,11.526658,NaN,NaN,NaN,NaN,NaN,...,NaN,0.161667,NaN,0.171778,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
# --------------------------------------------------------------------------
# Generate the synthetic dataset
# --------------------------------------------------------------------------
# Strategy: generate all prices first, then generate quantities using
# the full DGP with per-pair cross-price elasticities.

# Step 1: Generate prices and controls for all SKUs
price_data = {}

for product_id in product_list:
    weeks = np.arange(N_WEEKS)

    base_price = rng.uniform(200, 500)
    price_changes = rng.normal(0, 0.05, N_WEEKS).cumsum()
    log_price = np.log(base_price) + price_changes
    price = np.exp(log_price)

    seasonality = np.sin(2 * np.pi * weeks / 52)

    price_data[product_id] = {
        "price": price,
        "log_price": log_price,
        "seasonality": seasonality
    }

# Step 2: Generate quantities with the full DGP
all_rows = []

for product_id in product_list:
    params = true_params[product_id]
    pd_product = price_data[product_id]

    # Cross-price contribution: sum of gamma_ij * log(P_j) for each competitor
    cross_contribution = np.zeros(N_WEEKS)
    for comp_id, gamma_ij in true_cross_price[product_id].items():
        cross_contribution += gamma_ij * price_data[comp_id]["log_price"]

    log_demand = (
        params["intercept"]
        + params["elasticity"] * pd_product["log_price"]
        + cross_contribution
        + TRUE_BETA_SEASON * pd_product["seasonality"]
        + rng.normal(0, TRUE_SIGMA, N_WEEKS)
    )

    quantity = np.exp(log_demand).astype(int).clip(1)

    for t in range(N_WEEKS):
        all_rows.append({
            "week": t,
            "product_id": product_id,
            "product": params["product"],
            "category": params["category"],
            "state": params["state"],
            "price": round(pd_product["price"][t], 2),
            "quantity": quantity[t],
            "log_price": round(pd_product["log_price"][t], 4),
            "log_quantity": round(np.log(max(quantity[t], 1)), 4),
            "seasonality": round(pd_product["seasonality"][t], 4)
        })

df = pd.DataFrame(all_rows)
print(f"Dataset shape: {df.shape}")
print(f"Products: {df['product_id'].nunique()}, Weeks: {df['week'].nunique()}")
print(f"\nAll models fit the same 'log_quantity' column.")
print(f"Cross-price effects are baked into the DGP at the Product-pair level.")
df.head(10)

Dataset shape: (1872, 10)
Products: 18, Weeks: 104

All models fit the same 'log_quantity' column.
Cross-price effects are baked into the DGP at the Product-pair level.


,week,product_id,product,category,state,price,quantity,log_price,log_quantity,seasonality
0,0,Honda (Florida),Honda,Sedan,Florida,222.96,262810,5.4070,12.4792,0.0000
1,1,Honda (Florida),Honda,Sedan,Florida,224.56,261452,5.4141,12.4740,0.1205
2,2,Honda (Florida),Honda,Sedan,Florida,232.45,248932,5.4487,12.4249,0.2393
3,3,Honda (Florida),Honda,Sedan,Florida,227.53,288255,5.4273,12.5716,0.3546
4,4,Honda (Florida),Honda,Sedan,Florida,229.34,282239,5.4352,12.5505,0.4647
5,5,Honda (Florida),Honda,Sedan,Florida,236.63,261817,5.4665,12.4754,0.5681
6,6,Honda (Florida),Honda,Sedan,Florida,233.00,249802,5.4510,12.4284,0.6631
7,7,Honda (Florida),Honda,Sedan,Florida,238.38,295738,5.4739,12.5972,0.7485
8,8,Honda (Florida),Honda,Sedan,Florida,230.62,272656,5.4408,12.5160,0.8230
9,9,Honda (Florida),Honda,Sedan,Florida,226.47,278743,5.4226,12.5380,0.8855


In [52]:
# save master dataset to csv
df.to_csv('data/synthetic_dataset.csv', index=False)

In [ ]:
# ---------------------------------------------
# Prepare indices (shared across all models)
# ---------------------------------------------

cat_names = sorted(df["category"].unique())
state_names = sorted(df["state"].unique())
product_ids = sorted(df["product_id"].unique())

cat_idx_map = {c: i for i, c in enumerate(cat_names)}
state_idx_map = {r: i for i, r in enumerate(state_names)}
product_idx_map = {s: i for i, s in enumerate(product_ids)}

product_to_cat = np.array([cat_idx_map[true_params[s]["category"]] for s in product_ids])
product_to_state = np.array([state_idx_map[true_params[s]["state"]] for s in product_ids])

obs_product_idx = df["product_id"].map(product_idx_map).values

print(f"Categories: {cat_names}")
print(f"States: {state_names}")
print(f"Products: {len(product_ids)}")
print(f"Observations: {len(df)}")

Categories: ['LargeSUV', 'Sedan', 'SmallSUV']
States: ['Florida', 'NewYork']
Products: 18
Observations: 1872


In [20]:
# --------------------------------------------------------------------------
# Build cross-price feature matrix for Model 3
# --------------------------------------------------------------------------
# For each observation (product_id, week), we need the log-price of every
# competitor in the same category-region. We represent this as a flat
# list of (target_product, source_product) pairs with a feature matrix.

# Enumerate all cross-price pairs: (target_product_id, source_product_id)
cross_pairs = []
for product_id in product_ids:
    for comp_id in sorted(true_cross_price[product_id].keys()):
        cross_pairs.append((product_id, comp_id))

cross_pair_labels = [f"{true_params[t]['product']}←{true_params[s]['product']} ({true_params[t]['state']})"
                     for t, s in cross_pairs]

print(f"Total cross-price pairs: {len(cross_pairs)}")
print(f"\nPairs:")
for label, (t, s) in zip(cross_pair_labels, cross_pairs):
    gamma = true_cross_price[t][s]
    print(f"  {label:55s}  true γ = {gamma:.3f}")

# Build the feature matrix: shape (n_obs, n_pairs)
# For observation of product i at week t, column for pair (i, j) = log_price of j at t
# All other columns = 0 (pair doesn't apply to this observation)

n_obs = len(df)
n_pairs = len(cross_pairs)

# Pre-compute log-price arrays per product (indexed by week)
lp_lookup = {}
for product_id in product_ids:
    sub = df[df["product_id"] == product_id].sort_values("week")
    lp_lookup[product_id] = sub["log_price"].values

cross_price_matrix = np.zeros((n_obs, n_pairs))

for pair_idx, (target_id, source_id) in enumerate(cross_pairs):
    target_mask = (df["product_id"] == target_id).values
    source_lp = lp_lookup[source_id]
    target_weeks = df.loc[target_mask, "week"].values
    cross_price_matrix[target_mask, pair_idx] = source_lp[target_weeks]

print(f"\nCross-price feature matrix shape: {cross_price_matrix.shape}")
print(f"Non-zero entries: {(cross_price_matrix != 0).sum()} / {cross_price_matrix.size}")

# True gamma vector (same order as cross_pairs)
true_gamma_vector = np.array([true_cross_price[t][s] for t, s in cross_pairs])

Total cross-price pairs: 36

Pairs:
  Chevrolet←Mazda (Florida)                                true γ = 0.290
  Chevrolet←Volkswagen (Florida)                           true γ = 0.183
  Chevrolet←Mazda (NewYork)                                true γ = 0.258
  Chevrolet←Volkswagen (NewYork)                           true γ = 0.148
  Honda←Hyundai (Florida)                                  true γ = 0.348
  Honda←Toyota (Florida)                                   true γ = 0.333
  Honda←Hyundai (NewYork)                                  true γ = 0.334
  Honda←Toyota (NewYork)                                   true γ = 0.363
  Hyundai←Honda (Florida)                                  true γ = 0.365
  Hyundai←Toyota (Florida)                                 true γ = 0.361
  Hyundai←Honda (NewYork)                                  true γ = 0.337
  Hyundai←Toyota (NewYork)                                 true γ = 0.355
  Jeep←Porsche (Florida)                                   true γ = 0.063
  